In [ ]:
## Import necessary modules
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn import metrics
#from sklearn.metrics import plot_confusion_matrix
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.model_selection import RepeatedKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve, matthews_corrcoef
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sn
import seaborn as sns
import pandas as pd
import csv
import math
from sklearn.feature_selection import SelectKBest, chi2
from sklearn import preprocessing
from sklearn.model_selection import cross_val_score,KFold,cross_val_predict
from sklearn.metrics import confusion_matrix,classification_report,accuracy_score
import sklearn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/T Cell Antigen Datasets/Features/Features Extraxtion/DS-1/SMR_LPQ_features_T_Cell_Train_DS1.csv')

In [ ]:
# ============================================================
# Import necessary modules
# ============================================================

import numpy as np
import pandas as pd
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# ============================================================
# Handle missing and infinite values
# ============================================================

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(999, inplace=True)

# ============================================================
# Split features and labels
# ============================================================

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

print(f"Original data shape: {X.shape}")
print(f"Original class distribution: {Counter(y)}")


# ============================================================
# Train-test split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# ============================================================
# Feature scaling
# Fit only on training data, then transform train and test
# ============================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


print("\nBefore SMOTE:")
print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")
print(f"Training class distribution: {Counter(y_train)}")
print(f"Testing class distribution: {Counter(y_test)}")


# ============================================================
# Apply SMOTE only on training data
# ============================================================

smote = SMOTE(random_state=42)

X_train, y_train = smote.fit_resample(
    X_train,
    y_train
)

# ============================================================
# Final dataset after SMOTE
# ============================================================

print("\nAfter SMOTE:")
print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")
print(f"Training class distribution: {Counter(y_train)}")
print(f"Testing class distribution: {Counter(y_test)}")

print(f"\nFinal training shape: {X_train.shape}")
print(f"Final testing shape: {X_test.shape}")

DeepGLU Model

In [ ]:
import numpy as np
import tensorflow as tf
from keras import layers, regularizers
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, roc_auc_score
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.optimizers import Adam
from sklearn.model_selection import train_test_split, StratifiedKFold

import tensorflow as tf
from keras import layers, regularizers

class DenseSwiGLU(layers.Layer):
    def __init__(self, units, beta=10.0, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.beta = beta
        self.gate = layers.Dense(units, activation=None)
        self.value = layers.Dense(units, activation=None)

    def call(self, x):
        g = self.gate(x)
        gate_swish = g * tf.nn.sigmoid(self.beta * g)
        v = self.value(x)
        return gate_swish * v

import tensorflow as tf
from keras import layers, regularizers

def build_cnn_swiglu_model(input_dim, l1=1e-3, dropout=0.1, beta=10.0):

    inputs = tf.keras.Input(shape=(input_dim, 1))

    # ===== CNN block 1 =====
    x = layers.Conv1D(64,kernel_size=3,padding="same",kernel_regularizer=regularizers.l1(l1))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(dropout)(x)

    # ===== Flatten before GLU =====
    x = layers.Flatten()(x)

    # ===== SwiGLU block =====
    glu = DenseSwiGLU(units=32, beta=beta)(x)
    x = layers.Concatenate()([x, glu])

    # ===== Fully connected =====
    x = layers.Dense(32)(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(dropout)(x)

    outputs = layers.Dense(1, activation="sigmoid")(x)

    return tf.keras.Model(inputs, outputs)

model = build_cnn_swiglu_model(input_dim=X_train.shape[1], beta=10.0)

# Compile the model
learning_rate = 0.0001
optimizer = Adam(learning_rate=learning_rate)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# Initialize lists to store evaluation metrics for each fold
accuracy_list = []
f1_list = []
precision_list = []
recall_list = []
sensitivity_list = []
specificity_list = []
mcc_list = []
auc_list = []

# Perform K-fold cross-validation
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
from keras.callbacks import EarlyStopping, ReduceLROnPlateau


# 在K-fold循环中
for train_index, val_index in skf.split(X_train,y_train):
    X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
    y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]

    # 定义回调
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)

    callbacks = [early_stop, reduce_lr]

    # 训练
    history = model.fit(X_train_fold, y_train_fold,
                       epochs=20,
                       batch_size=32,
                       verbose=0,
                       #validation_split=0.1,
                       validation_data=(X_val_fold, y_val_fold),
                       callbacks=callbacks)

    # 保持您原有的评估代码不变
    y_val_pred = model.predict(X_val_fold, verbose=0).ravel()
    y_val_pred_binary = (y_val_pred > 0.5).astype(int)


    # Calculate confusion matrix for the current fold
    cm = confusion_matrix(y_val_fold, y_val_pred_binary)
    # Calculate AUC for the current fold
    auc = roc_auc_score(y_val_fold, y_val_pred)
    auc_list.append(auc)


    # Calculate performance evaluation metrics for the current fold
    TN = cm[0, 0]
    FP = cm[0, 1]
    FN = cm[1, 0]
    TP = cm[1, 1]

    # Accuracy
    accuracy = (TP + TN) / float(TP + TN + FP + FN)
    accuracy_list.append(accuracy)

    # F1-Score
    f1 = 2 * TP / float(2 * TP + FP + FN)
    f1_list.append(f1)

    # Precision
    precision = TP / float(TP + FP) if (TP + FP) != 0 else 0.0
    precision_list.append(precision)

    # Recall
    recall = TP / float(TP + FN)
    recall_list.append(recall)

    # Sensitivity
    sensitivity = TP / float(TP + FN)
    sensitivity_list.append(sensitivity)

    # Specificity
    specificity = TN / float(TN + FP)
    specificity_list.append(specificity)

    # MCC (Matthews Correlation Coefficient)
    mcc = ((TP * TN) - (FP * FN)) / float((np.sqrt((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN))) or 1)
    mcc_list.append(mcc)

# Calculate average performance evaluation metrics across all folds
avg_accuracy = np.mean(accuracy_list)
avg_f1 = np.mean(f1_list)
avg_precision = np.mean(precision_list)
avg_recall = np.mean(recall_list)
avg_sensitivity = np.mean(sensitivity_list)
avg_specificity = np.mean(specificity_list)
avg_mcc = np.mean(mcc_list)

# Calculate average AUC across all folds
avg_auc = np.mean(auc_list)
# Print average performance evaluation metrics
print("Accuracy =", avg_accuracy)
print("F1 Score =", avg_f1)
print("Precision =", avg_precision)
print("Recall =", avg_recall)
print("Sensitivity =", avg_sensitivity)
print("Specificity =", avg_specificity)
print("MCC =", avg_mcc)
print("AUC =", avg_auc)
